Generate Collect Earth Online plots for a SIDS country, following the CEO example plot GeoJSON.
Each plot is one cell of the 30m grid for the country's region from `get_gridspec`, so plots are exactly the GeoMAD and
LULC pixels. Plots are stratified by ESA WorldCover loaded onto that grid and mapped to our level 1 classes.

1. Load GADM for the country and buffer 200m (antimeridian fixed).
2. Load ESA WorldCover (level 1 classes) onto each 30m grid tile covering the buffered country and merge the tiles.
3. Choose the plots with `random_sampling` (equal stratified random). Classes with too few pixels give all they have,
   so the total can be a little under N_PLOTS.
4. Write a zipped shapefile in EPSG:4326 with a square polygon per plot, with PLOTID and ExtraInfo (the country name).
   The class is not written, so it does not bias the interpreters.

In [4]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import xarray as xr
from odc.geo.geom import Geometry

from ldn.grids import get_gridspec
from ldn.random_sampling import random_sampling
from ldn.training_data import LULC_PRODUCTS, get_buffered_country, load_and_prepare, product_nodata_value
from ldn.utils import DEP_COUNTRIES_AND_CODES, TRAINING_DATA_YEAR, WGS84, get_analysis_epsg

In [8]:
# country_of_interest = {"Fiji": "FJI"} # TODO: Test with antimeridian
country_of_interest = {"Anguilla": "AIA"}
BUFFER_M = 200
N_PLOTS = 3_000
SEED = 0
WORLDCOVER = LULC_PRODUCTS[0]

name = next(iter(country_of_interest))
PLOTS_PATH = Path(f"country_plots/{name}_plots.shp.zip")

In [ ]:
region = "pacific" if name in DEP_COUNTRIES_AND_CODES else "non-pacific"
analysis_crs = get_analysis_epsg(region)
grid = get_gridspec(region)
aoi = get_buffered_country(country_of_interest, analysis_crs, BUFFER_M).dissolve()
aoi_geom = Geometry(aoi.to_crs(analysis_crs).geometry.iloc[0], crs=analysis_crs)

# TODO: Maybe do this per tile. Then we won't hit memory issues.
# e.g. Kiribati spanning a huge part of the Pacific, or Fiji spanning the antimeridian.
# Then the tiles with low land cover will have a lot of plots in them.
# Is the WC LULC clipped to land?
# Then all tiles for a country can be merged simply.

tiles = [
    load_and_prepare(geobox, aoi, WORLDCOVER, TRAINING_DATA_YEAR) for _, geobox in grid.tiles_from_geopolygon(aoi_geom)
]
bands = [tile[WORLDCOVER["output_band"]] for tile in tiles]

# Tiles do not overlap, so write each into one nodata array spanning them all
# ponytail: dense array over the country's bounding box, memory grows with it (e.g. Fiji with Rotuma)
x = np.unique(np.concatenate([band.x.values for band in bands]))
y = np.unique(np.concatenate([band.y.values for band in bands]))[::-1]
wc = xr.DataArray(
    np.full((len(y), len(x)), product_nodata_value, dtype="uint8"), coords={"y": y, "x": x}, dims=("y", "x")
)
for band in bands:
    wc.loc[{"y": band.y, "x": band.x}] = band.values
wc = wc.rio.write_crs(analysis_crs)
print(f"{name}: merged {len(tiles)} tiles into {wc.shape}")

Anguilla: merged 4 tiles into (1755, 1626)


In [9]:
samples = random_sampling(
    da=wc.rename({"y": "latitude", "x": "longitude"}),
    n=N_PLOTS,
    min_sample_n=0,  # Not used by equal stratified random
    sampling="equal_stratified_random",
    drop_value=product_nodata_value,
    seed=SEED,
)
print(samples["lulc"].value_counts().sort_index())

# Shuffle so PLOTID order does not follow the class
samples = samples.sample(frac=1, random_state=SEED)
centres = samples.geometry.to_crs(analysis_crs)
plots = gpd.GeoDataFrame(
    {"PLOTID": np.arange(1, len(samples) + 1).astype(str), "ExtraInfo": name},
    geometry=centres.buffer(grid.resolution.x / 2, cap_style="square").to_crs(WGS84).values,
)
plots.to_file(PLOTS_PATH, driver="ESRI Shapefile")
print(f"Wrote {len(plots)} plots to {PLOTS_PATH}")

lulc
1    429
2    429
3    408
4    429
5    429
6    429
7    429
Name: count, dtype: int64
Wrote 2982 plots to country_plots/Anguilla_plots.shp.zip
